<a href="https://colab.research.google.com/github/freetechacctanup/code-with-codespaces/blob/main/first_agent_openai_cromadb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
pip install pypdf langchain langgraph langchain-openai langchain-community chromadb openai tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 6.4 MB/s eta 0:00:00


In [14]:
import os

os.environ["OPENAI_API_KEY"] = "sk-proj-KEY"

In [22]:
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# document path
PATH_TO_PDF_FILE = "/content/sample_data/procument.PDF"

# load document
doc_loader = PyPDFLoader(PATH_TO_PDF_FILE)
docs_data = doc_loader.load()

# split into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = splitter.split_documents(docs_data)

# lets create vec db - chroma db
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./qna_croma_db"
)

# retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# openai llm
model_name = "gpt-4o-mini"
llm = ChatOpenAI(model=model_name, temperature=0)

# prompts
prompt = ChatPromptTemplate.from_template(
    """
    You are a helpful assistant. Answer the question based only on the context
    below.
    If you don't know the answer, say 'I don't know.'

    Context: {context}

    Question: {question}
    """
)

doc_contents = "\n\n".join(doc.page_content for doc in docs_data)

rag_chain = (
    {
        "context": lambda x: (retriever | doc_contents),
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

# print(rag_chain)


In [23]:
# agentic flow
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

class QnAState(TypedDict):
  question:str
  context:str
  answer:str

def retrieve_node(state: QnAState):
  docs = retriever.invoke(state["question"])
  state["context"] = doc_contents
  return state

def generate(state: QnAState):
  response = llm.invoke(
      prompt.format_messages(
          context=state["context"],
          question=state["question"]
      )

  )
  state["answer"] = response.content

  return state

# Build graph
graph = StateGraph(QnAState)
graph.add_node("retriever", retrieve_node)
graph.add_node("generate", generate)

graph.set_entry_point("retriever")
graph.add_edge("retriever", "generate")
graph.add_edge("generate", END)

app = graph.compile()

result = app.invoke(
    {
        "question": "What is procurement policy answer in 3 sentece max"
    }
)

print(result["answer"])

Procurement policy refers to the set of principles and guidelines that govern the procurement process within an organization. It outlines the procedures for acquiring goods, services, and projects, ensuring compliance with legal and ethical standards. The policy aims to promote efficiency, transparency, and accountability in procurement operations.
